In [ ]:
# System & IO
import os
import glob
import joblib
import random

# Data Manipulation
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

# Statistics & Machine Learning Baselines
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_squared_error, r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor

# PyTorch & Geometric
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
from torch_geometric.loader import DataLoader
from torch.utils.data import Dataset

# Visualization & Progress
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm, trange
import seaborn as sns
sns.set(style='whitegrid')

# Hardware Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)





In [ ]:
# --- GLOBAL CONSTANTS ---
# The primary root for all data
BASE_DIR = '../Data/Models/Model_1' 

# Directory paths
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
PROCESSED_TRAIN_DIR = os.path.join(TRAIN_DIR, 'Processed')
PROCESSED_VAL_DIR = os.path.join(BASE_DIR, 'val', 'Processed')
PROCESSED_TEST_DIR = os.path.join(BASE_DIR, 'internal_test', 'Processed')

# Static Node Files
DF_1D_STATIC_PATH = os.path.join(TRAIN_DIR, '1d_nodes_static.csv')
DF_2D_STATIC_PATH = os.path.join(TRAIN_DIR, '2d_nodes_static.csv')

# Edge & Connection Files
PATH_1D_EDGES = os.path.join(TRAIN_DIR, '1d_edge_index.csv')
PATH_2D_EDGES = os.path.join(TRAIN_DIR, '2d_edge_index.csv')
PATH_CONNECTIONS = os.path.join(TRAIN_DIR, '1d2d_connections.csv')

# Search Patterns (Optional, for globbing)
TRAIN_PATTERN = os.path.join(PROCESSED_TRAIN_DIR, '*.pt')
VAL_PATTERN   = os.path.join(PROCESSED_VAL_DIR, '*.pt')
TEST_PATTERN  = os.path.join(PROCESSED_TEST_DIR, '*.pt')

# Join the directory and the search pattern in one go
TRAIN_PATTERN = os.path.join(BASE_DIR, 'train', 'Processed', '*.pt')
VAL_PATTERN   = os.path.join(BASE_DIR, 'val', 'Processed', '*.pt')
TEST_PATTERN  = os.path.join(BASE_DIR, 'internal_test', 'Processed', '*.pt')

# Model Save Paths
model_1_save_path = 'best_flood_model_1.pth'
model_2_save_path = 'best_flood_model_2.pth'

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load the pre-processed HeteroData object from disk
        return torch.load(self.file_paths[idx])

In [ ]:

# 1. Collect all processed .pt file paths for each split
# Using glob to grab every file in the Processed directories we just created
train_files = glob.glob(TRAIN_PATTERN)
val_files = glob.glob(VAL_PATTERN)
test_files = glob.glob(TEST_PATTERN)

# 2. Initialize the Datasets
# Assuming your FloodDataset class is designed to take a list of file paths
train_ds = FloodDataset(train_files)
val_ds = FloodDataset(val_files)
test_ds = FloodDataset(test_files)

# 3. Create the Loaders
# shuffle=True for training to ensure the model doesn't learn the order of storms
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# Quick sanity check for your report
print("Loaders initialized:")
print(f" - Training Batches: {len(train_loader)}")
print(f" - Validation Batches: {len(val_loader)}")
print(f" - Test Batches: {len(test_loader)}")

### Baseline Model, Random Forest

In [ ]:
def prepare_tabular_data(loader):
    """
    Flattens HeteroData objects into X (features) and y (targets).
    For the baseline, we predict the 'next' timestep for each node.
    """
    X_list, y_list = [], []
    
    for data in loader:
        # data['node1d'].x shape is [Timesteps, Nodes, Features]
        # We'll focus on 1D nodes for this example baseline
        x_1d = data['node1d'].x.numpy()
        
        # Create sliding window: Use time 't' to predict 't+1'
        # Features: [t], Target: [t+1, water_depth_index]
        # Assuming water depth is the last dynamic feature added
        X_frames = x_1d[:-1, :, :] # All steps except last
        y_frames = x_1d[1:, :, -1:] # All steps except first, depth only
        
        # Flatten [Timesteps * Nodes, Features]
        X_flat = X_frames.reshape(-1, X_frames.shape[-1])
        y_flat = y_frames.reshape(-1)
        
        X_list.append(X_flat)
        y_list.append(y_flat)
        
    return np.vstack(X_list), np.concatenate(y_list)

In [ ]:
print("Flattening data for Random Forest...")
X_train, y_train = prepare_tabular_data(train_loader)
X_val, y_val = prepare_tabular_data(val_loader)

rf_baseline = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)

print("Training Random Forest Baseline...")
rf_baseline.fit(X_train, y_train)

# 3. Evaluate
y_pred = rf_baseline.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Baseline Validation RMSE: {rmse:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch

# Configuration
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

COLORS = {
    'input': '#1a252f',      
    'baseline': '#566573',   
    'primary': '#1f618d',    
    'output': '#a04000',     
    'bg': '#ffffff',         
    'text_light': '#ffffff',
    'text_dark': '#1a252f'
}

# Adjusted width/height for maximum efficiency
fig, ax = plt.subplots(figsize=(16, 4.5), facecolor=COLORS['bg'])
# Tightened xlim (0 to 14 instead of 16) to remove side gaps
ax.set_xlim(0, 14)
ax.set_ylim(0, 4.5) 
ax.axis('off')

def draw_compact_box(x, y, w, h, title, subtitle, color):
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05,rounding_size=0.05", 
                          linewidth=0, facecolor=color, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h*0.62, title, ha='center', va='center', 
            fontsize=13, fontweight='black', color=COLORS['text_light'], zorder=3)
    ax.text(x + w/2, y + h*0.32, subtitle, ha='center', va='center', 
            fontsize=10, fontweight='bold', color=COLORS['text_light'], zorder=3)

# Title - Positioned to minimize top whitespace
ax.text(7, 4.1, "SYSTEM ARCHITECTURE: TOPOLOGICAL VS. STATISTICAL MODELING", 
        fontsize=18, fontweight='black', ha='center', color=COLORS['text_dark'])

# 1. Shared Input (Left)
draw_compact_box(0.2, 1.2, 2.8, 1.6, "1. SOURCE DATA", 
                "Heterogeneous Graph\nNodes, Edges, Features", COLORS['input'])

# 2. Baseline Pathway (Middle Top)
draw_compact_box(5.0, 2.3, 4.0, 1.3, "BASELINE BENCHMARK", 
                "Random Forest (Topology-Blind)", COLORS['baseline'])

# 3. Proposed Pipeline (Middle Bottom)
draw_compact_box(5.0, 0.4, 4.0, 1.3, "PROPOSED: FloodGNN", 
                "ST-GNN (Topology-Aware)", COLORS['primary'])

# 4. Evaluation/Output (Right)
draw_compact_box(11.0, 1.2, 2.8, 1.6, "2. EVALUATION", 
                "Water Level Predictions\nRMSE / $R^2$ Score", COLORS['output'])

# Thicker arrows
arrow_props = dict(arrowstyle='-|> ,head_width=0.4,head_length=0.6', color='#34495e', lw=2.5)

# Connectors
ax.annotate('', xy=(5.0, 2.95), xytext=(3.0, 2.0), arrowprops=dict(connectionstyle="angle,angleA=0,angleB=90,rad=5", **arrow_props))
ax.annotate('', xy=(5.0, 1.05), xytext=(3.0, 2.0), arrowprops=dict(connectionstyle="angle,angleA=0,angleB=-90,rad=5", **arrow_props))
ax.annotate('', xy=(11.0, 2.0), xytext=(9.0, 2.95), arrowprops=dict(connectionstyle="angle,angleA=0,angleB=-90,rad=5", **arrow_props))
ax.annotate('', xy=(11.0, 2.0), xytext=(9.0, 1.05), arrowprops=dict(connectionstyle="angle,angleA=0,angleB=90,rad=5", **arrow_props))

# Labels
ax.text(4.0, 3.1, "Data Flattening", fontsize=10, fontweight='bold', ha='center', style='italic')
ax.text(4.0, 0.7, "Graph Maintained", fontsize=10, fontweight='bold', ha='center', style='italic')

# Final Save with absolute minimal padding
plt.savefig('architecture_no_whitespace.png', dpi=300, bbox_inches='tight', pad_inches=0.05)
plt.show()

### Primary Model

In [ ]:
class FloodGNN(nn.Module):
    def __init__(self, hidden_channels, out_channels, dropout_p=0.2):
        super().__init__()
        self.dropout_p = dropout_p
        
        # 1. Spatial Layers
        self.conv1 = HeteroConv({
            ('node1d', 'pipe', 'node1d'): SAGEConv((-1, -1), hidden_channels),
            ('node2d', 'surface', 'node2d'): SAGEConv((-1, -1), hidden_channels),
            ('node1d', 'exchange', 'node2d'): SAGEConv((-1, -1), hidden_channels),
            ('node2d', 'exchange', 'node1d'): SAGEConv((-1, -1), hidden_channels),
        }, aggr='sum')

        self.conv2 = HeteroConv({
            ('node1d', 'pipe', 'node1d'): SAGEConv(hidden_channels, hidden_channels),
            ('node2d', 'surface', 'node2d'): SAGEConv(hidden_channels, hidden_channels),
            ('node1d', 'exchange', 'node2d'): SAGEConv(hidden_channels, hidden_channels),
            ('node2d', 'exchange', 'node1d'): SAGEConv(hidden_channels, hidden_channels),
        }, aggr='sum')

        # 2. Temporal Layers
        self.gru_1d = nn.GRU(hidden_channels, hidden_channels, batch_first=True)
        self.gru_2d = nn.GRU(hidden_channels, hidden_channels, batch_first=True)

        # 3. Output Heads
        self.dropout = nn.Dropout(p=dropout_p)
        self.lin_1d = nn.Linear(hidden_channels, out_channels)
        self.lin_2d = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, h_dict=None):
        # --- Layer 1 ---
        h_dict_spatial = self.conv1(x_dict, edge_index_dict)
        h_dict_spatial = {key: torch.relu(h) for key, h in h_dict_spatial.items()}
        h_dict_spatial = {key: self.dropout(h) for key, h in h_dict_spatial.items()}

        # --- Layer 2 ---
        h_dict_spatial = self.conv2(h_dict_spatial, edge_index_dict)
        h_dict_spatial = {key: torch.relu(h) for key, h in h_dict_spatial.items()}
        h_dict_spatial = {key: self.dropout(h) for key, h in h_dict_spatial.items()}
        
        # --- Temporal Pass ---
        # node1d
        x_1d = h_dict_spatial['node1d'].unsqueeze(1)
        h_1d, next_h1d = self.gru_1d(x_1d, h_dict['node1d'] if h_dict else None)
        
        # node2d
        x_2d = h_dict_spatial['node2d'].unsqueeze(1)
        h_2d, next_h2d = self.gru_2d(x_2d, h_dict['node2d'] if h_dict else None)

        # Apply dropout one last time before the final linear layer
        out_1d = self.lin_1d(self.dropout(h_1d.squeeze(1)))
        out_2d = self.lin_2d(self.dropout(h_2d.squeeze(1)))
        
        return out_1d, out_2d, {'node1d': next_h1d, 'node2d': next_h2d}

In [ ]:
def calculate_dataset_std(loader):
    all_1d_levels = []
    all_2d_levels = []
    
    print("Calculating dataset statistics...")
    for batch in tqdm(loader):
        # Extract the target water level (last column of the feature matrix)
        # Shape of x: [Timesteps, Nodes, Features]
        y_1d = batch['node1d'].x[:, :, -1].flatten()
        y_2d = batch['node2d'].x[:, :, -1].flatten()
        
        all_1d_levels.append(y_1d.numpy())
        all_2d_levels.append(y_2d.numpy())
    
    # Concatenate all events into one giant array
    full_1d = np.concatenate(all_1d_levels)
    full_2d = np.concatenate(all_2d_levels)
    
    # Calculate Standard Deviation
    std_1d = np.std(full_1d)
    std_2d = np.std(full_2d)
    
    return std_1d, std_2d

In [ ]:
std_1d, std_2d = calculate_dataset_std(train_loader)

# Create your own dictionary for the loss function
std_dev_dict = {
    (1, 1): std_1d, 
    (1, 2): std_2d,
    (2, 1): std_1d, # If Model 2 is trained on the same data, use the same std
    (2, 2): std_2d
}

print(f"\nCalculated 1D Std: {std_1d:.4f}")
print(f"Calculated 2D Std: {std_2d:.4f}")

In [ ]:
class StandardizedRMSELoss(nn.Module):
    def __init__(self, std_dev_dict):
        super().__init__()
        self.std_dev_dict = std_dev_dict
    
    def forward(self, pred_1d, pred_2d, target_1d, target_2d, model_id):
        # Get std devs for this model
        std_1d = self.std_dev_dict[(model_id, 1)]
        std_2d = self.std_dev_dict[(model_id, 2)]
        
        # Calculate RMSE for each node type
        rmse_1d = torch.sqrt(torch.mean((pred_1d - target_1d) ** 2) + 1e-6)
        rmse_2d = torch.sqrt(torch.mean((pred_2d - target_2d) ** 2) + 1e-6)
        
        # Standardize by dividing by std dev
        std_rmse_1d = rmse_1d / std_1d
        std_rmse_2d = rmse_2d / std_2d
        
        # Average across node types
        return (std_rmse_1d + std_rmse_2d) / 2

In [ ]:
def validate_event_with_metrics(model, event_data, criterion, model_id):
    model.eval()
    h_dict = None
    num_t = event_data['node1d'].x.size(0)
    current_x_1d = event_data['node1d'].x[0]
    current_x_2d = event_data['node2d'].x[0]
    
    total_loss = 0
    all_preds = []
    all_targets = []

    for t in range(1, num_t):
        x_dict = {'node1d': current_x_1d, 'node2d': current_x_2d}
        with torch.no_grad():
            pred_1d, pred_2d, h_dict = model(x_dict, event_data.edge_index_dict, h_dict)
        
        target_1d = event_data['node1d'].x[t, :, -1].unsqueeze(-1)
        target_2d = event_data['node2d'].x[t, :, -1].unsqueeze(-1)
        
        total_loss += criterion(pred_1d, pred_2d, target_1d, target_2d, model_id).item()

        # Collect for residuals (using 1D as the primary metric for example)
        all_preds.append(pred_1d.cpu().numpy().flatten())
        all_targets.append(target_1d.cpu().numpy().flatten())

        # Autoregressive step
        next_x_1d = event_data['node1d'].x[t].clone()
        next_x_1d[:, -1] = pred_1d.squeeze()
        next_x_2d = event_data['node2d'].x[t].clone()
        next_x_2d[:, -1] = pred_2d.squeeze()
        current_x_1d, current_x_2d = next_x_1d, next_x_2d

    return total_loss / (num_t - 1), np.concatenate(all_preds), np.concatenate(all_targets)

def train_event(model, event_data, optimizer, criterion, model_id, 
                teacher_forcing_ratio=0.5, max_grad_norm=1.0):
    model.train()
    event_data = event_data.to(device)
    optimizer.zero_grad()
    
    num_t = event_data['node1d'].x.size(0)
    accumulated_loss = 0 
    running_loss_val = 0.0
    
    h_dict = None 
    current_x_1d = event_data['node1d'].x[0]
    current_x_2d = event_data['node2d'].x[0]

    for t in range(1, num_t):
        x_dict = {'node1d': current_x_1d, 'node2d': current_x_2d}
        pred_1d, pred_2d, h_dict = model(x_dict, event_data.edge_index_dict, h_dict)
        
        target_1d = event_data['node1d'].x[t, :, -1].unsqueeze(-1)
        target_2d = event_data['node2d'].x[t, :, -1].unsqueeze(-1)
        
        loss = criterion(pred_1d, pred_2d, target_1d, target_2d, model_id)
        accumulated_loss = accumulated_loss + loss
        running_loss_val += loss.item()
        
        # Teacher Forcing Logic
        if random.random() < teacher_forcing_ratio:
            current_x_1d = event_data['node1d'].x[t]
            current_x_2d = event_data['node2d'].x[t]
        else:
            # Autoregressive: feedback the prediction into the next time step
            next_x_1d = event_data['node1d'].x[t].clone()
            next_x_1d[:, -1] = pred_1d.detach().squeeze()
            next_x_2d = event_data['node2d'].x[t].clone()
            next_x_2d[:, -1] = pred_2d.detach().squeeze()
            current_x_1d, current_x_2d = next_x_1d, next_x_2d

    # Optimization
    accumulated_loss.backward()
    # Tunable gradient clipping to prevent exploding gradients in recurrent loops
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
    optimizer.step()
    
    return running_loss_val / (num_t - 1)


def train_model(model, train_loader, val_loader, optimizer, criterion, model_id, 
                num_epochs=20, initial_tf=0.5, tf_decay=0.05, min_tf=0.1):
    
    train_losses, val_losses = [], []
    r2_history = []
    best_val_loss = float('inf')
    SAVE_PATH = f'best_model_{model_id}.pth'
    
    # residuals are stored only for the validation set during training
    val_residuals = None 

    for epoch in range(num_epochs):
        current_tf = max(min_tf, initial_tf - (epoch * tf_decay))
        
        # --- TRAINING PHASE ---
        model.train()
        total_train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
            total_train_loss += train_event(model, batch, optimizer, criterion, model_id, current_tf)
        
        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # --- VALIDATION PHASE (Used for tuning/checkpointing) ---
        epoch_preds, epoch_targets = [], []
        total_val_loss = 0
        # No gradients here, and STRICTLY using val_loader
        for batch in val_loader:
            v_loss, p, t = validate_event_with_metrics(model, batch.to(device), criterion, model_id)
            total_val_loss += v_loss
            epoch_preds.append(p)
            epoch_targets.append(t)
        
        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        # Calculate R2 for the validation set
        full_preds = np.concatenate(epoch_preds)
        full_targets = np.concatenate(epoch_targets)
        current_r2 = r2_score(full_targets, full_preds)
        r2_history.append(current_r2)

        # Save checkpoint if this is the best validation performance yet
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), SAVE_PATH)
            val_residuals = full_targets - full_preds # Store best residuals
            print(f" >>> Best model saved (Val Loss: {avg_val_loss:.4f})")

        print(f"E{epoch+1} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | R2: {current_r2:.4f}")

    tf_history = {
        'r2': r2_history,
        'residuals': val_residuals,
        'tf_ratios': [max(min_tf, initial_tf - (i * tf_decay)) for i in range(num_epochs)]
    }
    
    return train_losses, val_losses, tf_history

def validate_event(model, event_data, criterion, model_id):
    """Simplified version of train_event without backprop or teacher forcing"""
    h_dict = None
    num_t = event_data['node1d'].x.size(0)
    current_x_1d = event_data['node1d'].x[0]
    current_x_2d = event_data['node2d'].x[0]
    total_loss = 0

    for t in range(1, num_t):
        x_dict = {'node1d': current_x_1d, 'node2d': current_x_2d}
        pred_1d, pred_2d, h_dict = model(x_dict, event_data.edge_index_dict, h_dict)
        
        target_1d = event_data['node1d'].x[t, :, -1].unsqueeze(-1)
        target_2d = event_data['node2d'].x[t, :, -1].unsqueeze(-1)
        
        total_loss += criterion(pred_1d, pred_2d, target_1d, target_2d, model_id).item()

        # Autoregressive step: always use predictions for validation
        next_x_1d = event_data['node1d'].x[t].clone()
        next_x_1d[:, -1] = pred_1d.squeeze()
        next_x_2d = event_data['node2d'].x[t].clone()
        next_x_2d[:, -1] = pred_2d.squeeze()
        current_x_1d, current_x_2d = next_x_1d, next_x_2d

    return total_loss / (num_t - 1)

def finalize_test_metrics(model, test_loader, criterion, model_id, save_path):
    # Load the best state found during validation
    model.load_state_dict(torch.load(save_path))
    model.eval()
    
    test_preds, test_targets = [], []
    total_test_loss = 0
    
    for batch in test_loader:
        loss, p, t = validate_event_with_metrics(model, batch.to(device), criterion, model_id)
        total_test_loss += loss
        test_preds.append(p)
        test_targets.append(t)
        
    avg_test_loss = total_test_loss / len(test_loader)
    all_p = np.concatenate(test_preds)
    all_t = np.concatenate(test_targets)
    test_r2 = r2_score(all_t, all_p)
    
    print(f"\n--- FINAL TEST RESULTS ---")
    print(f"Test Loss: {avg_test_loss:.4f}")
    print(f"Test R2: {test_r2:.4f}")
    
    return avg_test_loss, test_r2, (all_t - all_p)

In [ ]:
class WeightedStandardizedRMSELoss(nn.Module):
    def __init__(self, std_dev_dict, weight_1d=0.7):
        super().__init__()
        self.std_dev_dict = std_dev_dict
        self.weight_1d = weight_1d
        self.weight_2d = 1.0 - weight_1d

    def forward(self, pred_1d, pred_2d, target_1d, target_2d, model_id):
        # Get std devs for this model [cite: 589, 592]
        std_1d = self.std_dev_dict[(model_id, 1)]
        std_2d = self.std_dev_dict[(model_id, 2)]
        
        # Calculate RMSE for each node type [cite: 594]
        rmse_1d = torch.sqrt(torch.mean((pred_1d - target_1d) ** 2) + 1e-6)
        rmse_2d = torch.sqrt(torch.mean((pred_2d - target_2d) ** 2) + 1e-6)
        
        # Standardize [cite: 595, 597]
        std_rmse_1d = rmse_1d / std_1d
        std_rmse_2d = rmse_2d / std_2d
        
        # Weighted average to prioritize sewer (1D) performance
        return (self.weight_1d * std_rmse_1d) + (self.weight_2d * std_rmse_2d)

In [ ]:
# --- Improved Hardware & Model Setup ---
model_1 = FloodGNN(hidden_channels=32, out_channels=1, dropout_p=0.3).to(device) # Increased dropout [cite: 494, 711]

# Higher weight decay to combat the overfitting seen in initial logs [cite: 715, 779]
optimizer_1 = torch.optim.Adam(model_1.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = WeightedStandardizedRMSELoss(std_dev_dict, weight_1d=0.7)

# --- Improved Training Strategy ---
# Slower decay gives the model more time to adjust to its own errors [cite: 648]
INITIAL_TF = 0.5    
TF_DECAY = 0.02     # Slower weaning (previously 0.05) [cite: 718]
MIN_TF = 0.2        # Higher floor for sequence stability [cite: 719, 722]
NUM_EPOCHS = 30     # Increased epochs to allow regularization to settle [cite: 723]

history_1 = train_model(
    model=model_1, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    optimizer=optimizer_1, 
    criterion=criterion, 
    model_id=1, 
    num_epochs=NUM_EPOCHS, 
    initial_tf=INITIAL_TF, 
    tf_decay=TF_DECAY, 
    min_tf=MIN_TF
)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import r2_score
import joblib

def evaluate_model_comprehensive(model, loader, device, model_name="Model", save_path=None):
    """
    Consolidates evaluation: Loads weights, runs autoregressive inference, 
    calculates metrics, and returns a dictionary ready for scaling/plotting.
    """
    if save_path:
        model.load_state_dict(torch.load(save_path))
        print(f"Loaded weights from {save_path}")
        
    model.eval()
    results = {
        '1d': {'preds': [], 'actuals': []},
        '2d': {'preds': [], 'actuals': []}
    }

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Evaluating {model_name}"):
            batch = batch.to(device)
            # Assuming batch has a time dimension at index 0
            num_t = batch['node1d'].x.size(0)
            h_dict = None
            
            # Initial state (t=0)
            curr_x1d, curr_x2d = batch['node1d'].x[0], batch['node2d'].x[0]

            for t in range(1, num_t):
                x_dict = {'node1d': curr_x1d, 'node2d': curr_x2d}
                p1d, p2d, h_dict = model(x_dict, batch.edge_index_dict, h_dict)
                
                # Store results for metrics
                results['1d']['preds'].append(p1d.cpu().numpy())
                results['1d']['actuals'].append(batch['node1d'].x[t, :, -1].cpu().numpy())
                
                results['2d']['preds'].append(p2d.cpu().numpy())
                results['2d']['actuals'].append(batch['node2d'].x[t, :, -1].cpu().numpy())
                
                # Autoregressive Update: Prediction becomes input for t+1
                next_x1d = batch['node1d'].x[t].clone()
                next_x1d[:, -1] = p1d.squeeze()
                next_x2d = batch['node2d'].x[t].clone()
                next_x2d[:, -1] = p2d.squeeze()
                curr_x1d, curr_x2d = next_x1d, next_x2d

    # Flatten for global metrics
    p1d_all = np.concatenate([p.flatten() for p in results['1d']['preds']])
    a1d_all = np.concatenate([a.flatten() for a in results['1d']['actuals']])
    p2d_all = np.concatenate([p.flatten() for p in results['2d']['preds']])
    a2d_all = np.concatenate([a.flatten() for a in results['2d']['actuals']])
    
    rmse_1d = np.sqrt(np.mean((p1d_all - a1d_all)**2))
    rmse_2d = np.sqrt(np.mean((p2d_all - a2d_all)**2))
    combined_r2 = r2_score(np.concatenate([a1d_all, a2d_all]), np.concatenate([p1d_all, p2d_all]))
    
    print(f"\n--- {model_name} Results ---")
    print(f"1D RMSE: {rmse_1d:.4f} | 2D RMSE: {rmse_2d:.4f} | R2: {combined_r2:.4f}")

    # This dictionary is the "source of truth" for the next steps
    eval_metrics = {
        'preds_1d': results['1d']['preds'],
        'preds_2d': results['2d']['preds'],
        'residuals': (a1d_all - p1d_all),
        'r2_final': combined_r2,
        'rmse_1d': rmse_1d,
        'rmse_2d': rmse_2d,
        'model_name': model_name
    }
    return eval_metrics

def get_original_scale_predictions(eval_metrics, scaler_1d_path, scaler_2d_path):
    """
    Inverse transforms standardized predictions back to physical units.
    """
    s1d = joblib.load(scaler_1d_path)
    s2d = joblib.load(scaler_2d_path)

    # Water level is at the last index
    idx = -1
    scale1, mean1 = s1d.scale_[idx], s1d.mean_[idx]
    scale2, mean2 = s2d.scale_[idx], s2d.mean_[idx]

    # Apply x = (z * sigma) + mu and clip physically impossible negatives
    preds_1d_phys = [np.clip(p * scale1 + mean1, 0, None) for p in eval_metrics['preds_1d']]
    preds_2d_phys = [np.clip(p * scale2 + mean2, 0, None) for p in eval_metrics['preds_2d']]

    print(f"--- {eval_metrics['model_name']} Physical Scale ---")
    print(f"1D Range: {np.min([p.min() for p in preds_1d_phys]):.3f}m to {np.max([p.max() for p in preds_1d_phys]):.3f}m")
    
    return {'1d_m': preds_1d_phys, '2d_m': preds_2d_phys}

def plot_learning_curve(history_tuple, model_name):
    """
    history_tuple: (train_losses, val_losses, eval_metrics)
    """
    train_losses, val_losses, eval_metrics = history_tuple
    epochs = range(1, len(train_losses) + 1)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))

    ax1.plot(epochs, train_losses, label='Train')
    ax1.plot(epochs, val_losses, label='Val', linestyle='--')
    ax1.set_title(f'{model_name}: Loss')
    ax1.legend()

    gap = np.array(val_losses) / np.array(train_losses)
    ax2.plot(epochs, gap, color='red')
    ax2.axhline(y=1, color='black', linestyle=':')
    ax2.set_title('Overfitting Ratio (Val/Train)')

    if 'residuals' in eval_metrics:
        ax3.hist(eval_metrics['residuals'], bins=50, color='purple', alpha=0.7)
        ax3.axvline(x=0, color='black')
        ax3.set_title('Residual Distribution (Std units)')

    plt.tight_layout()
    plt.show()

In [ ]:
eval_results = evaluate_model_comprehensive(
    model_1, val_loader, device, 
    model_name="GNN_v1", 
    save_path='best_model_1.pth'
)

# 2. Plot results (Assuming history_1 = [train_loss_list, val_loss_list])
plot_learning_curve((history_1[0], history_1[1], eval_results), "GNN_v1")

# 3. Get Physical Scale Values
physical_preds = get_original_scale_predictions(
    eval_results, 'scaler_dyn_1d.pkl', 'scaler_dyn_2d.pkl'
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch

# Configuration
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

COLORS = {
    'input': '#1a252f',      
    'conv': '#1f618d',       
    'process': '#6c3483',    
    'output': '#a04000',     
    'highlight': '#1e8449',  
    'bg': '#ffffff',         
    'text_light': '#ffffff',
    'text_dark': '#1a252f'
}

# Tightened width to 13.5 to kill the right-side gap
fig, ax = plt.subplots(figsize=(14, 4.0), facecolor=COLORS['bg'])
ax.set_xlim(0, 13.5)
ax.set_ylim(0, 4.2) 
ax.axis('off')

def draw_compact_box(x, y, w, h, title, subtitle, color):
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05,rounding_size=0.05", 
                          linewidth=0, facecolor=color, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h*0.65, title, ha='center', va='center', 
            fontsize=12, fontweight='black', color=COLORS['text_light'], zorder=3)
    ax.text(x + w/2, y + h*0.35, subtitle, ha='center', va='center', 
            fontsize=9.5, fontweight='bold', color=COLORS['text_light'], zorder=3)

# Title
ax.text(6.75, 3.8, "FLOODGNN: INTEGRATED SPATIO-TEMPORAL PIPELINE", 
        fontsize=17, fontweight='black', ha='center', color=COLORS['input'])

# 1. Input Section
draw_compact_box(0.1, 1.0, 2.8, 1.8, "1. GRAPH INPUT", 
                "1D/2D Node Features\nMulti-Relational Edges", COLORS['input'])

# 2. Convolution + Processing
draw_compact_box(3.5, 1.0, 3.2, 1.8, "2. HETEROCONV", 
                "SAGE Message Passing\nReLU + Dropout (0.3)", COLORS['conv'])

# 3. Temporal Engine 
draw_compact_box(7.3, 1.0, 3.2, 1.8, "3. TEMPORAL ENGINE", 
                "Autoregressive Loop\nTeacher Forcing / Clipping", COLORS['highlight'])

# 4. Output Heads (Shifted to the absolute edge)
draw_compact_box(11.1, 1.0, 2.3, 1.8, "4. PREDICTION", 
                "1D: Water Level\n2D: Flood Depth", COLORS['output'])

# Arrows
arrow_props = dict(arrowstyle='-|> ,head_width=0.3,head_length=0.5', color='#34495e', lw=2)
ax.annotate('', xy=(3.5, 1.9), xytext=(2.9, 1.9), arrowprops=arrow_props)
ax.annotate('', xy=(7.3, 1.9), xytext=(6.7, 1.9), arrowprops=arrow_props)
ax.annotate('', xy=(11.1, 1.9), xytext=(10.5, 1.9), arrowprops=arrow_props)

# Feedback Loop
ax.annotate('Recurrence', xy=(8.0, 1.0), xytext=(9.8, 0.6), 
            arrowprops=dict(connectionstyle="arc3,rad=0.4", **arrow_props),
            fontsize=9, fontweight='bold', ha='center')

# Bottom Spec Bar
spec_text = "ADAM ($10^{-3}$)  |  LATENT DIM: 32  |  PARAMS: ~2.4k"
ax.text(6.75, 0.15, spec_text, ha='center', fontsize=10, fontweight='bold', 
        color=COLORS['text_dark'], bbox=dict(facecolor='white', alpha=1.0, edgecolor='#bdc3c7', boxstyle='round,pad=0.2'))

# Force tight layout on save
plt.savefig('flood_gnn_final_tight.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()